**Question 1**

In [1]:
import pandas as pd
import requests
from io import StringIO
import re

In [ ]:
# Data Loading (from iposcoop.com)

url = "https://www.iposcoop.com/ipos-recently-filed"
 
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/91.0.4472.124 Safari/537.36'
    )
}
 
try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Use StringIO buffer
    df_ipos = pd.read_html(StringIO(response.text))[0]
    print(f"Successfully loaded IPO data. Shape: {df_ipos.shape}")
    print(f"\nColumn names:\n{df_ipos.columns.tolist()}\n")
    
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

Successfully loaded IPO data. Shape: (500, 10)

Column names:
['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']



In [ ]:
# filter withdrawn IPOs

withdrawn_df = df_ipos[df_ipos['Expected To Trade'] == 'Withdrawn'].copy()
print(f"Number of withdrawn IPOs: {len(withdrawn_df)}")
withdrawn_df.head()

Number of withdrawn IPOs: 34


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
17,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O
20,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O
32,2026-09-15,OTSAW Ltd. (Withdrawn),OTSA,Aegis Capital,4.40,$4.50,$5.50,$22.00,Withdrawn,S/O
38,2026-09-10,Motive Technologies (Withdrawn),MTVE,J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...,0.00,NaN,NaN,$100.00,Withdrawn,S/O
43,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,R.F. Lafferty & Co.,2.00,$4.00,$5.00,$9.00,Withdrawn,S/O


In [ ]:
# Company Classification

def classify_company_type(company_name):
    """
    Categorize company based on patterns (order matters).
    Returns the first matched category.
    """
    if pd.isna(company_name):
        return 'Other'
    
    name = str(company_name)
    
    # Order matters: check in this sequence
    if 'Technologies' in name:
        return 'Technologies'
    elif 'Acquisition Corp' in name or 'Acquisition Corporation' in name or 'Corp' in name:
        return 'Acquisition Corp'
    elif 'Inc' in name or 'Incorporated' in name:
        return 'Inc.'
    elif 'Group' in name:
        return 'Group'
    elif 'Ltd' in name or 'Limited' in name:
        return 'Limited'
    elif 'Holdings' in name or 'Holding' in name:
        return 'Holdings'
    else:
        return 'Other'
 
withdrawn_df['Company Type'] = withdrawn_df['Company'].apply(classify_company_type)
print(f"\nCompany Type distribution:")
print(withdrawn_df['Company Type'].value_counts())


Company Type distribution:
Company Type
Limited             8
Other               8
Group               5
Acquisition Corp    5
Holdings            4
Technologies        3
Inc.                1
Name: count, dtype: int64


In [11]:
# Price Parsing

def extract_numeric(price_str):
    """
    Extract numeric value from strings like '$8.00' or '-'
    Returns float or NaN for missing values
    """
    if pd.isna(price_str):
        return None
    
    price_str = str(price_str).strip()
    
    # Handle missing values
    if price_str == '-' or price_str == '':
        return None
    
    # Remove all non-numeric characters except decimal point
    cleaned = re.sub(r'[^\d.]', '', price_str)
    
    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None

withdrawn_df['Price_Low'] = withdrawn_df['Price Low'].apply(extract_numeric)
withdrawn_df['Price_High'] = withdrawn_df['Price High'].apply(extract_numeric)
withdrawn_df['Avg_price'] = (withdrawn_df['Price_Low'] + withdrawn_df['Price_High']) / 2.0

withdrawn_df[['Company', 'Price Low', 'Price High', 'Avg_price']].head(50)

,Company,Price Low,Price High,Avg_price
17,Rise Smart Group (Withdrawn),$4.00,$4.00,4.00
20,C2 Capital Group (Withdrawn),$4.00,$5.00,4.50
32,OTSAW Ltd. (Withdrawn),$4.50,$5.50,5.00
38,Motive Technologies (Withdrawn),NaN,NaN,NaN
43,Idea Tech Holding (Withdrawn),$4.00,$5.00,4.50
50,Hornbeck Offshore Services (Withdrawn),NaN,NaN,NaN
52,Coolbit Technologies Ltd. (Withdrawn),$4.00,$5.00,4.50
59,Timber Road Acquisition Corp. (Withdrawn),$10.00,$10.00,10.00
69,pan-Africa Corp. (Withdrawn),$10.00,$10.00,10.00
84,Living Homeopathy (Withdrawn),$4.00,$6.00,5.00


In [15]:
# Numeric Conversion

withdrawn_df['Shares_millions'] = pd.to_numeric(
    withdrawn_df['Shares (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

withdrawn_df['Est_Vol_millions'] = pd.to_numeric(
    withdrawn_df['Est $ Vol (millions)'].astype(str).str.replace('$', '').str.replace(',', ''),
    errors='coerce'
)

In [16]:
# Value Calculation

withdrawn_df['Shares_offered_value'] = (withdrawn_df['Shares_millions'] * withdrawn_df['Avg_price']).fillna(withdrawn_df['Est_Vol_millions'])

In [17]:
# Aggregation
result = withdrawn_df.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)

print("="*60)
print("WITHDRAWAL VALUES BY COMPANY TYPE (in $ millions)")
print("="*60)
print(result)
print()
print(f"Answer: {result.idxmax()} - ${result.max():,.2f} million")

WITHDRAWAL VALUES BY COMPANY TYPE (in $ millions)
Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             219.2500
Technologies        184.9000
Group                56.5750
Name: Shares_offered_value, dtype: float64

Answer: Acquisition Corp - $499.99 million
